# Hydraulic System Condition Monitoring: Backpropagation vs Predictive Coding

**Domain:** Hydraulic-system condition monitoring  
**Target:** Internal pump leakage severity: 0 = no leakage, 1 = weak leakage, 2 = severe leakage  
**Architecture:** Three hidden layers (32, 16, 8 units), then three output classes.

This notebook compares a standard backpropagation MLP with one explicit predictive-coding-style implementation. It demonstrates cycle-level sensor feature engineering, layer-wise predictions and errors, latent-state inference, total energy, and held-out classification metrics.

This is an educational experiment on a laboratory test rig, not a validated industrial or safety-critical maintenance system.

## 1. Engineering problem and dataset

The UCI *Condition monitoring of hydraulic systems* dataset contains 2,205 repeated load cycles from an experimental hydraulic test rig. Measurements include pressure, motor power, volume flow, temperature, vibration, and cooling/efficiency signals. Cycle-wise labels include the internal pump leakage class.

**Question:** Can measurements collected during a hydraulic operating cycle distinguish no leakage, weak leakage, and severe leakage?

We summarize each sensor's cycle using mean, standard deviation, minimum, maximum, and median. This converts variable-rate raw measurements into a fixed-length feature vector. It is interpretable feature engineering, but it discards temporal ordering and waveform shape.

**Citation:** Helwig, N., Pignanelli, E., & Schütze, A. (2015). *Condition monitoring of hydraulic systems*. UCI Machine Learning Repository. https://doi.org/10.24432/C5CW21 (CC BY 4.0, per UCI).

## 2. Install packages and download data

The notebook downloads the UCI archive automatically. If that URL changes, download the archive from https://archive.ics.uci.edu/dataset/447/condition+monitoring+of+hydraulic+systems and extract it into `/content/hydraulic_data/`.

In [ ]:
!pip -q install numpy pandas scikit-learn matplotlib torch

In [ ]:
from pathlib import Path
import urllib.request, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_ROOT = Path("/content/hydraulic_data")
DATA_ROOT.mkdir(parents=True, exist_ok=True)
archive = DATA_ROOT / "hydraulic_systems.zip"
url = "https://archive.ics.uci.edu/static/public/447/condition+monitoring+of+hydraulic+systems.zip"

if not list(DATA_ROOT.rglob("profile.txt")):
    try:
        urllib.request.urlretrieve(url, archive)
        with zipfile.ZipFile(archive, "r") as zf:
            zf.extractall(DATA_ROOT)
        print("Downloaded and extracted UCI data.")
    except Exception as exc:
        print("Automatic download failed:", repr(exc))
        print("Download the UCI archive manually, extract into", DATA_ROOT, "and rerun.")

profiles = list(DATA_ROOT.rglob("profile.txt"))
if not profiles:
    raise FileNotFoundError("profile.txt missing. Download/extract the UCI archive into DATA_ROOT.")
PROFILE_PATH = profiles[0]
DATA_DIR = PROFILE_PATH.parent
print("Data directory:", DATA_DIR)

## 3. Read sensor cycles and engineer features

Each row of a sensor file corresponds to a cycle, and its columns are samples during that cycle. `profile.txt` is aligned by cycle. Its columns include cooler condition, valve condition, pump leakage, accumulator pressure, and a stability flag. Only pump leakage is used as the target; profile labels are not model inputs.

In [ ]:
SENSORS = ["PS1","PS2","PS3","PS4","PS5","PS6","EPS1","FS1","FS2",
           "TS1","TS2","TS3","TS4","VS1","CE","CP","SE"]
STAT_NAMES = ["mean","std","min","max","median"]

def read_matrix(path):
    df = pd.read_csv(path, sep="\\t", header=None, engine="python")
    df = df.apply(pd.to_numeric, errors="coerce").dropna(axis=1, how="all")
    return df.to_numpy(dtype=np.float64)

profile = read_matrix(PROFILE_PATH)
if profile.shape[1] < 5:
    raise ValueError(f"Expected 5 profile columns, got {profile.shape}")
y_raw = profile[:, 2].astype(int)
stable_flag = profile[:, 4].astype(int)

blocks, feature_names, used_sensors = [], [], []
for sensor in SENSORS:
    paths = list(DATA_DIR.rglob(f"{sensor}.txt"))
    if not paths:
        print("Skipping missing sensor:", sensor)
        continue
    arr = read_matrix(paths[0])
    if arr.shape[0] != len(profile):
        raise ValueError(f"{sensor}: {arr.shape[0]} rows but profile has {len(profile)}")
    stats = np.column_stack([
        np.nanmean(arr, axis=1), np.nanstd(arr, axis=1),
        np.nanmin(arr, axis=1), np.nanmax(arr, axis=1), np.nanmedian(arr, axis=1)
    ])
    blocks.append(stats)
    feature_names.extend([f"{sensor}_{s}" for s in STAT_NAMES])
    used_sensors.append(sensor)

if not blocks:
    raise RuntimeError("No sensor files found. Check archive extraction.")
X_all = np.concatenate(blocks, axis=1)
valid = np.isfinite(X_all).all(axis=1) & np.isin(y_raw, [0,1,2])
X = X_all[valid].astype(np.float32)
y = y_raw[valid].astype(np.int64)
stable = stable_flag[valid]
class_names = {0:"No leakage", 1:"Weak leakage", 2:"Severe leakage"}

print("Profile cycles:", len(profile))
print("Sensors used:", used_sensors)
print("Feature matrix:", X.shape)
print("Target distribution:", dict(zip(*np.unique(y, return_counts=True))))
print("Stable flag counts:", dict(zip(*np.unique(stable, return_counts=True))))

In [ ]:
counts = pd.Series(y).value_counts().sort_index()
plt.figure(figsize=(7,4))
plt.bar([class_names[i] for i in counts.index], counts.values)
plt.ylabel("Number of cycles")
plt.title("Pump leakage class distribution")
plt.xticks(rotation=10)
plt.tight_layout()
plt.show()
display(pd.DataFrame(X, columns=feature_names).head())

## 4. Split and preprocessing

We use a stratified 80/20 hold-out split and fit the scaler on training data only. Both models receive the same split and scaled features.

**Generalization caveat:** Randomly split cycles may be highly similar across train and test. Treat this as a teaching baseline, not proof of generalization to unseen operating conditions. A group-/condition-aware split is a recommended follow-up.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

SEED = 42
Xtr_raw, Xte_raw, ytr_np, yte_np = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)
scaler = StandardScaler()
Xtr_np = scaler.fit_transform(Xtr_raw).astype(np.float32)
Xte_np = scaler.transform(Xte_raw).astype(np.float32)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_loader = DataLoader(
    TensorDataset(torch.tensor(Xtr_np), torch.tensor(ytr_np, dtype=torch.long)),
    batch_size=64, shuffle=True
)
print("Device:", device, "| train:", Xtr_np.shape, "| test:", Xte_np.shape)

## 5. Model A — standard backpropagation

Architecture: engineered features → 32 → 16 → 8 → 3. The three hidden layers use ReLU. The output gives three class logits. Cross-entropy is differentiated through the full network, and Adam updates all weights and biases.

In [ ]:
class BackpropMLP(nn.Module):
    def __init__(self, input_dim, widths=(32,16,8), output_dim=3):
        super().__init__()
        layers, prev = [], input_dim
        for width in widths:
            layers += [nn.Linear(prev, width), nn.ReLU()]
            prev = width
        layers.append(nn.Linear(prev, output_dim))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

torch.manual_seed(SEED)
bp = BackpropMLP(Xtr_np.shape[1]).to(device)
bp_opt = torch.optim.Adam(bp.parameters(), lr=1e-3, weight_decay=1e-4)
loss_fn = nn.CrossEntropyLoss()
BP_EPOCHS = 60
bp_history = []
for epoch in range(BP_EPOCHS):
    bp.train()
    total, n = 0.0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        bp_opt.zero_grad()
        loss = loss_fn(bp(xb), yb)
        loss.backward()
        bp_opt.step()
        total += loss.item() * len(xb); n += len(xb)
    bp_history.append(total/n)

plt.figure(figsize=(7,4))
plt.plot(bp_history, label="Backprop training cross-entropy")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend(); plt.tight_layout(); plt.show()

## 6. Model B — predictive coding with three hidden layers

Let the observed feature vector be \(z_0=x\), hidden latent states be \(z_1,z_2,z_3\) (dimensions 32, 16, 8), and the one-hot target be the clamped state \(z_4=y\) during training.

Each layer predicts the next state: \(μ_l=f_l(W_l z_{l-1}+b_l)\). Hidden predictions use ReLU; the output prediction uses softmax. Errors are \(ε_l=z_l-μ_l\), layer energies are \(E_l=0.5\,mean(||ε_l||^2)\), and total energy is \(E=Σ_l E_l\).

For each minibatch:
1. Initialize latent states with a feedforward pass.
2. Clamp inputs and training targets.
3. Iteratively update hidden latent states to reduce total energy while weights are fixed.
4. Update each layer's weights from its own local energy while inferred states are fixed.

This is one explicit predictive-coding-style formulation, not the only formulation. Its objective includes hidden-layer prediction errors, while the backpropagation baseline minimizes output cross-entropy. The notebook compares the two learning procedures, not identical optimization objectives.

In [ ]:
class PredictiveCodingMLP(nn.Module):
    def __init__(self, input_dim, widths=(32,16,8), output_dim=3):
        super().__init__()
        dims = [input_dim, *widths, output_dim]
        self.layers = nn.ModuleList([nn.Linear(dims[i], dims[i+1]) for i in range(len(dims)-1)])
    def predict_layer(self, idx, previous):
        raw = self.layers[idx](previous)
        return torch.relu(raw) if idx < len(self.layers)-1 else torch.softmax(raw, dim=-1)
    def initialize_states(self, x):
        states = [x]
        for idx in range(3):
            states.append(self.predict_layer(idx, states[-1]).detach())
        return states
    def errors_energies(self, states, y_onehot):
        z_all = [*states, y_onehot]
        mus, errors, energies = [], [], []
        for idx in range(4):
            mu = self.predict_layer(idx, z_all[idx])
            eps = z_all[idx+1] - mu
            energy = 0.5 * (eps**2).sum(dim=1).mean()
            mus.append(mu); errors.append(eps); energies.append(energy)
        return mus, errors, energies

def infer_states(model, x, y_onehot, steps=12, state_lr=0.08):
    with torch.no_grad():
        init = model.initialize_states(x)
    latent = [init[i].detach().clone().requires_grad_(True) for i in (1,2,3)]
    for _ in range(steps):
        _, _, energies = model.errors_energies([x,*latent], y_onehot)
        total_energy = torch.stack(energies).sum()
        grads = torch.autograd.grad(total_energy, latent)
        with torch.no_grad():
            for z, g in zip(latent, grads):
                z -= state_lr * g
                z.clamp_(min=0.0)
        latent = [z.detach().requires_grad_(True) for z in latent]
    return [x, *[z.detach() for z in latent]]

def pc_predict(model, x):
    # No target at test time: use feedforward initialization and output prediction.
    with torch.no_grad():
        states = model.initialize_states(x)
        return model.predict_layer(3, states[3])

torch.manual_seed(SEED)
pc = PredictiveCodingMLP(Xtr_np.shape[1]).to(device)
pc_opts = [torch.optim.Adam(layer.parameters(), lr=5e-4, weight_decay=1e-4) for layer in pc.layers]
PC_EPOCHS, PC_STEPS, PC_STATE_LR = 40, 12, 0.08
pc_history = []
for epoch in range(PC_EPOCHS):
    pc.train(); total, n = 0.0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        yoh = torch.eye(3, device=device)[yb]
        states = infer_states(pc, xb, yoh, PC_STEPS, PC_STATE_LR)
        # Local parameter learning: each layer uses its own energy.
        for idx, opt in enumerate(pc_opts):
            opt.zero_grad()
            z_all = [*states, yoh]
            mu = pc.predict_layer(idx, z_all[idx])
            eps = z_all[idx+1] - mu
            local_energy = 0.5 * (eps**2).sum(dim=1).mean()
            local_energy.backward()
            opt.step()
        with torch.no_grad():
            _, _, energies = pc.errors_energies(states, yoh)
            total += torch.stack(energies).sum().item() * len(xb)
            n += len(xb)
    pc_history.append(total/n)
    if (epoch+1) % 5 == 0:
        print(f"Epoch {epoch+1}/{PC_EPOCHS}: mean total energy={pc_history[-1]:.4f}")

plt.figure(figsize=(7,4))
plt.plot(pc_history, label="PC total energy (includes hidden-layer energies)")
plt.xlabel("Epoch"); plt.ylabel("Total energy"); plt.legend(); plt.tight_layout(); plt.show()

## 7. Inspect layer-wise prediction errors and energy

This cell exposes the intermediate quantities required by the project: each layer's predicted state, prediction error, layer energy, and total system energy. During this diagnostic, the target is clamped because the example comes from training data.

In [ ]:
pc.eval()
xb = torch.tensor(Xtr_np[:64], device=device)
yb = torch.tensor(ytr_np[:64], device=device)
yoh = torch.eye(3, device=device)[yb]
states = infer_states(pc, xb, yoh, PC_STEPS, PC_STATE_LR)
with torch.no_grad():
    mus, errors, energies = pc.errors_energies(states, yoh)
for i, (mu, eps, energy) in enumerate(zip(mus, errors, energies), start=1):
    print(f"Layer {i}: prediction shape={tuple(mu.shape)}, error shape={tuple(eps.shape)}, energy={energy.item():.6f}")
print("Total system energy:", torch.stack(energies).sum().item())

## 8. Held-out comparison

Accuracy and macro-F1 are reported alongside class-wise precision/recall and confusion matrices. Macro-F1 gives each class equal weight, which helps reveal poor performance on less frequent classes.

In [ ]:
bp.eval()
with torch.no_grad():
    bp_pred = bp(torch.tensor(Xte_np, device=device)).argmax(1).cpu().numpy()
pc.eval()
with torch.no_grad():
    pc_pred = pc_predict(pc, torch.tensor(Xte_np, device=device)).argmax(1).cpu().numpy()

results = pd.DataFrame([
    {"Model":"Backpropagation MLP", "Accuracy":accuracy_score(yte_np,bp_pred),
     "Macro-F1":f1_score(yte_np,bp_pred,average="macro",zero_division=0)},
    {"Model":"Predictive Coding MLP", "Accuracy":accuracy_score(yte_np,pc_pred),
     "Macro-F1":f1_score(yte_np,pc_pred,average="macro",zero_division=0)}
])
display(results.style.format({"Accuracy":"{:.3f}","Macro-F1":"{:.3f}"}))
print("BACKPROP CLASSIFICATION REPORT")
print(classification_report(yte_np,bp_pred,labels=[0,1,2],target_names=[class_names[i] for i in range(3)],zero_division=0))
print("PREDICTIVE CODING CLASSIFICATION REPORT")
print(classification_report(yte_np,pc_pred,labels=[0,1,2],target_names=[class_names[i] for i in range(3)],zero_division=0))

fig, axes = plt.subplots(1,2,figsize=(11,4))
for ax, pred, title in [(axes[0],bp_pred,"Backpropagation"),(axes[1],pc_pred,"Predictive Coding")]:
    ConfusionMatrixDisplay(confusion_matrix(yte_np,pred,labels=[0,1,2]),
        display_labels=[class_names[i] for i in range(3)]).plot(ax=ax,colorbar=False,values_format="d")
    ax.set_title(title)
plt.tight_layout(); plt.show()

## 9. Architecture and parameter check

In [ ]:
def parameter_count(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Input features:", Xtr_np.shape[1])
print("Hidden layers: 32, 16, 8 (three hidden layers)")
print("Output classes: 3")
print("Backprop learned parameters:", parameter_count(bp))
print("PC learned parameters:", parameter_count(pc))
print("PC latent states are temporary inferred variables, not learned weights.")

## 10. Discussion guide and limitations

Use actual outputs from this run in the report. Do not invent metrics or assume predictive coding will win.

- Backpropagation minimizes output cross-entropy by differentiating through the whole network.
- This PC formulation explicitly minimizes layer-wise prediction energies and infers hidden states iteratively.
- PC total energy is not directly comparable to backprop cross-entropy because PC energy includes hidden-layer prediction errors.
- Differences can arise from objective choice, inference schedule, learning rates, and optimization stability.
- Research concerns about gradient transport, weight symmetry, update locking, and biological plausibility are motivations for alternative learning rules; this notebook does not prove PC resolves all of them.
- This dataset comes from one laboratory rig. Random splitting may overestimate generalization to unseen operating conditions.
- Cycle statistics discard temporal order. A stronger follow-up should test a condition-aware/group-aware split, multiple random seeds, and severe-leakage recall.
- This is not validated for safety-critical maintenance.

**Dataset reference:** Helwig, N., Pignanelli, E., & Schütze, A. (2015). *Condition monitoring of hydraulic systems*. UCI Machine Learning Repository. https://doi.org/10.24432/C5CW21